# 🫀 실험 9b — lead-agnostic이 **저유도에서 무너지는 것**을 고친다

**MedKOS / `notebooks/exp9b_rlm_lowlead_fix.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 세 번 재현된 문제

한 가중치로 웨어러블·텔레메트리를 다 서비스하려고 Random Lead Masking(RLM)으로
lead-agnostic 모델을 학습했습니다. 결과가 **비대칭**입니다:

| agnostic − 유도고정 | 실험10 QUICK | 실험10 full | **실험10′ (CV)** |
|---|---|---|---|
| `{II}` (웨어러블) | −0.178 | −0.188 | **−0.0735** |
| `{12}` (텔레메트리) | −0.008 | +0.011 | **+0.0013** |

**세 번 모두 같은 방향**입니다. 크기는 클래스 균형을 잡으면서 줄었지만 구조는 그대로예요:

> **"한 가중치로 둘 다"는 텔레메트리 쪽만 공짜입니다.**
> 이 프로젝트의 주 배치는 **웨어러블**인데 거기서 −0.0735입니다.

## 왜 그럴까 — 세울 수 있는 가설 둘

**H-데이터**: RLM이 배치마다 구성을 **균등하게**(각 1/4) 고릅니다. `{12}`는 정보가
많아 손실이 빨리 줄고, `{II}`는 어려운데 4번에 1번만 나옵니다.
→ **최적화가 쉬운 쪽으로 쏠린다.**

**H-용량**: 마스크 벡터를 헤드에서 `Dense(16)`으로 붙이기만 했습니다. 컨볼루션 특징
자체는 모든 구성이 **공유**합니다. `{II}`와 `{12}`가 필요로 하는 특징이 다르면
공유 특징이 타협안이 되고, 그 타협은 **어려운 쪽이 손해**를 봅니다.

두 가설은 서로 배타적이지 않고, **고치는 방법이 다릅니다.**

## 처방 두 개 — 각각 한 가설을 겨눈다

| arm | 무엇을 바꾸나 | 겨누는 가설 |
|---|---|---|
| `uniform` | (기준선) 구성 균등 샘플링 | — · **실험10′에서 재사용** |
| **`weighted`** | 어려운 구성을 더 자주 뽑는다 (`{II}` 40% · `{I,II}` 25% · `{II,V1}` 25% · `{12}` 10%) | **H-데이터** |
| **`film`** | 마스크가 **컨볼루션 특징을 직접 변조**(FiLM: scale·shift) | **H-용량** |

FiLM은 채널마다 `γ(mask)·x + β(mask)`를 적용합니다. 헤드에서 정보를 얹는 게 아니라
**특징 추출 자체를 구성별로 다르게** 만드는 것이라 H-용량을 정면으로 겨눕니다.

## 비용 — 실험10′의 산출물을 재사용해서 절반으로

**유도고정 4개와 uniform agnostic은 다시 학습하지 않습니다.** 실험10′이 겹마다
저장한 arm 확률(`fixed_*_f{k}`, `agno_*_f{k}`)을 그대로 읽습니다.

```
실험10′  : 5모델 × 5겹 = 25회 학습
실험9b   : 2변형 × 5겹 = 10회 학습   ← 새로 도는 건 이것뿐
```

정렬이 깨지면 비교가 통째로 거짓이 되므로, **겹 배정·학습/검증 분할·언더샘플링
시드를 실험10′과 한 글자도 다르지 않게** 재현하고 CELL 4에서 대조 검증합니다.

## 사전등록

| # | 예측 | 판정 기준 |
|---|---|---|
| **G0 (관문)** | 재사용한 uniform의 `{II}` 손실이 실험10′과 일치 | 재현 실패면 정렬이 깨진 것 → **중단** |
| **H1 (주가설)** | `weighted` 또는 `film`의 `{II}` 손실이 `uniform`보다 **작다**(덜 나쁘다), 유의 | 부트스트랩 CI |
| **H2** | 그 개선이 `{12}`를 희생시키지 않는다 | `{12}` 손실이 −0.02 이내 유지 |
| **H3 (성공 기준)** | 어느 한 변형이 **유도고정 `{II}` 대비 −0.05 이내** | 퀘스트 큐가 정한 9b의 합격선 |
| **H4 (기전)** | `weighted`가 이기면 H-데이터, `film`이 이기면 H-용량 | 둘 다 이기면 원인이 둘 다 |

**H1이 깨지면**(둘 다 개선 없음) 그건 실패가 아니라 정보입니다 — 저유도 손실이
샘플링·용량 문제가 아니라 **정보량의 본질적 한계**라는 뜻이고, 그러면 배치 전략을
"한 가중치"에서 **"구성별 별도 모델"** 로 바꿔야 합니다.


In [ ]:
# CELL 1 — 설정 (★ 실험10′과 동일해야 하는 값들)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 아래 블록은 실험10′ CELL 1과 **한 글자도 달라선 안 된다**.
#     하나라도 다르면 겹 배정이나 학습 분할이 어긋나 재사용한 arm과 정렬이 깨진다.
FS = 100
CLASSES = ["NORM", "CD", "STTC", "MI", "HYP"]
G_FRONT = ["NORM", "CD", "STTC"]
G_MIXED = ["MI"]
G_TRANS = ["HYP"]
LEADS12 = ["I", "II", "III", "AVR", "AVL", "AVF",
           "V1", "V2", "V3", "V4", "V5", "V6"]
CONFIGS = {"II": [1], "I+II": [0, 1], "II+V1": [1, 6], "12": list(range(12))}
BALANCE_RATIO = 4.0
K_FOLD  = 5
N_SEEDS = 1
EPOCHS  = 20
SEED0   = 20260801
BOOT    = 2000
# ★★ 여기까지

# 9b가 새로 정하는 것: 난이도 비례 샘플링 확률 (어려운 구성을 더 자주)
SAMPLE_P = {"II": 0.40, "I+II": 0.25, "II+V1": 0.25, "12": 0.10}
assert abs(sum(SAMPLE_P.values()) - 1.0) < 1e-9
VARIANTS = ["weighted", "film"]     # uniform은 실험10′에서 재사용

CONFIG = dict(exp="exp9b_rlm_lowlead_fix", quest="ailab-2026-0015",
              parent_exp="exp10p_lead_cv",
              problem="lead-agnostic이 {II}에서 -0.0735, {12}에서 +0.0013 (세 번 재현)",
              hypotheses={"H-데이터": "균등 샘플링이 쉬운 구성으로 최적화를 쏠리게 함",
                          "H-용량": "컨볼루션 특징을 모든 구성이 공유해 어려운 쪽이 손해"},
              arms={"uniform": "실험10′ 재사용(기준선)",
                    "weighted": f"난이도 비례 샘플링 {SAMPLE_P}",
                    "film": "마스크가 컨볼루션 특징을 직접 변조(scale·shift)"},
              prediction=("G0 uniform 재현 · H1 {II} 손실 개선 유의 · "
                          "H2 {12} 희생 없음 · H3 유도고정 {II} 대비 -0.05 이내"),
              reuse="유도고정 4개 + uniform agnostic은 재학습 안 함",
              sample_p=SAMPLE_P, balance_ratio=BALANCE_RATIO, k_fold=K_FOLD,
              n_seeds=N_SEEDS, epochs=EPOCHS, fs=FS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp9b_rlm_fix", CONFIG, project=PROJECT)

# ── 실험10′ 실행 폴더를 registry.jsonl에서 찾는다
REG = os.path.join(PROJECT, "registry.jsonl")
prev = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp10p_lead_cv" and os.path.isdir(r.get("dir", "")):
        prev = r
if prev is None:
    raise RuntimeError("registry.jsonl에서 exp10p_lead_cv를 못 찾았습니다 — 실험10′을 먼저 돌리세요")
PREV = prev["dir"]
run.log(f"실험10′ 산출물: {PREV}")
run.log(f"  요약: {prev.get('summary','')}")

def prev_arm(name):
    p = os.path.join(PREV, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

missing = [f"{pre}_{c}_f{k}" for pre in ("fixed", "agno") for c in CONFIGS
           for k in range(K_FOLD) if prev_arm(f"{pre}_{c}_f{k}") is None]
if missing:
    raise RuntimeError(f"실험10′의 arm이 없습니다: {missing[:5]} … (총 {len(missing)}개)")
run.log(f"✅ 재사용할 arm {2*len(CONFIGS)*K_FOLD}개 확인 — 새로 학습할 것은 "
        f"{len(VARIANTS)}변형 × {K_FOLD}겹 = {len(VARIANTS)*K_FOLD}회뿐")


In [ ]:
# CELL 2 — 데이터 (실험10′의 full 캐시 재사용)
import wfdb, pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv", "scp_statements.csv"):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)

agg = scp[scp.diagnostic == 1].diagnostic_class.to_dict()
df["sc"] = df.scp_codes.apply(
    lambda s: sorted({agg[k] for k in ast.literal_eval(s) if k in agg}))
sub = df[df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)].copy()
sub["y"] = sub.sc.apply(lambda s: CLASSES.index(s[0]))

CACHE = run.data("ptbxl_12lead_full.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"캐시가 없습니다: {CACHE} — 실험10 또는 10′을 먼저 돌려 만드세요")
z = np.load(CACHE, allow_pickle=True)
X, Y, FOLD10, PID, EID = z["X"], z["y"], z["fold"], z["pid"], z["eid"]
if len(X) != len(sub) or not np.array_equal(np.sort(EID), np.sort(sub.index.values)):
    raise RuntimeError("캐시가 지금 sub와 다르다")
sub = sub.loc[EID]
run.log(f"X{X.shape} · 클래스 {np.bincount(Y, minlength=5).tolist()} ({CLASSES})")

CV = (FOLD10 - 1) % K_FOLD          # ★ 실험10′과 동일한 겹 배정
run.log(f"{K_FOLD}겹 배정 (실험10′과 동일)")
for k in range(K_FOLD):
    run.log(f"  겹 {k}: {(CV==k).sum():6,}건 · "
            f"클래스 {np.bincount(Y[CV==k], minlength=5).tolist()}")


### CELL 3 — 두 변형 학습 (uniform은 실험10′ 재사용)

`weighted`는 데이터 쪽, `film`은 모델 쪽을 건드립니다. **둘을 동시에 바꾸지 않는 이유**는
어느 가설이 맞는지 갈라야 하기 때문입니다 — 같이 바꾸면 좋아져도 원인을 모릅니다.


In [ ]:
# CELL 3 — weighted / film 변형 학습
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score

NC = len(CLASSES)

def mask_of(cfg):
    m = np.zeros(12, "float32"); m[CONFIGS[cfg]] = 1.0
    return m

def build_agnostic(seed, mode):
    """mode='plain'  : 마스크를 헤드에서만 합침 (실험10′과 동일 구조)
       mode='film'   : 마스크가 각 컨볼루션 블록의 특징을 scale·shift로 변조"""
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12)); mi = layers.Input((12,))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        if mode == "film":
            # FiLM: γ(mask)·x + β(mask). (batch, f) → (batch, 1, f) 로 브로드캐스트.
            g = layers.Reshape((1, f))(layers.Dense(f, activation="sigmoid")(mi))
            b = layers.Reshape((1, f))(layers.Dense(f)(mi))
            x = layers.Add()([layers.Multiply()([x, g]), b])
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Concatenate()([h, layers.Dense(16, activation="relu")(mi)])
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model([si, mi], layers.Dense(NC, activation="softmax")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
    return m

class RLMSeq(tf.keras.utils.Sequence):
    """probs=None이면 균등(실험10′과 동일), 아니면 그 분포로 구성을 뽑는다."""
    def __init__(self, idx, bs, seed, sw, probs=None, shuffle=True):
        self.idx, self.bs, self.shuffle, self.sw = idx, bs, shuffle, sw
        self.names = list(CONFIGS)
        self.p = None if probs is None else np.array([probs[n] for n in self.names])
        self.rs = np.random.RandomState(seed)
        self.order = idx.copy(); self.used = []
        self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.idx) / self.bs))
    def on_epoch_end(self):
        if self.shuffle: self.rs.shuffle(self.order)
    def __getitem__(self, i):
        b = self.order[i * self.bs:(i + 1) * self.bs]
        if self.shuffle:
            j = self.rs.choice(len(self.names), p=self.p)
            cfg = self.names[j]; self.used.append(cfg)
        else:
            cfg = self.names[i % len(self.names)]
        m = mask_of(cfg)
        return (X[b].astype("float32") * m, np.repeat(m[None], len(b), 0)), Y[b], self.sw[b]

def auto_weights(y, beta=0.9999):
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(NC)}
    return {c: float(v / w[0]) for c, v in w.items()}

def balance(idx, ratio, seed):
    rs = np.random.RandomState(seed)
    cnt = np.bincount(Y[idx], minlength=NC); cap = int(cnt[cnt > 0].min() * ratio)
    out = []
    for c in range(NC):
        ci = idx[Y[idx] == c]
        out.append(rs.choice(ci, cap, replace=False) if len(ci) > cap else ci)
    return np.sort(np.concatenate(out))

OOF = {f"{v}_{c}": np.zeros((len(Y), NC)) for v in VARIANTS for c in CONFIGS}
t0, done, total = time.time(), 0, len(VARIANTS) * K_FOLD * N_SEEDS
sampling_log = {}

for k in range(K_FOLD):
    # ★★ 실험10′ CELL 6과 **완전히 동일한** 분할·균형 절차
    te = CV == k
    rest = np.where(~te)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    va_i, tr_i_raw = rest[:n_val], rest[n_val:]
    te_i = np.where(te)[0]
    tr_i = balance(tr_i_raw, BALANCE_RATIO, SEED0 + k)
    CW = auto_weights(Y[tr_i]); SWALL = np.zeros(len(Y), "float32")
    SWALL[tr_i] = [CW[int(c)] for c in Y[tr_i]]
    run.log(f"\n── 겹 {k}: 학습 {len(tr_i):,} / 검증 {len(va_i):,} / 테스트 {len(te_i):,}")

    for v in VARIANTS:
        if all(run.load_arm(f"{v}_{c}_f{k}") is not None for c in CONFIGS):
            for c in CONFIGS:
                OOF[f"{v}_{c}"][te_i] = run.load_arm(f"{v}_{c}_f{k}")
            done += N_SEEDS
            run.log(f"  ⏭ {v} 이미 완료(체크포인트)")
            continue
        acc = {c: np.zeros((len(te_i), NC)) for c in CONFIGS}
        for s in range(N_SEEDS):
            probs = SAMPLE_P if v == "weighted" else None
            mode = "film" if v == "film" else "plain"
            m = build_agnostic(SEED0 + 100 * k + 70 + s, mode)
            seq = RLMSeq(tr_i, 128, SEED0 + k + s, SWALL, probs=probs)
            m.fit(seq, validation_data=RLMSeq(va_i, 256, SEED0, SWALL, shuffle=False),
                  epochs=EPOCHS, verbose=0)
            if v == "weighted":
                from collections import Counter
                sampling_log[f"f{k}"] = dict(Counter(seq.used))
            for c in CONFIGS:
                mk = mask_of(c)
                acc[c] += m.predict([X[te_i].astype("float32") * mk,
                                     np.repeat(mk[None], len(te_i), 0)],
                                    batch_size=512, verbose=0)
            if k == 0 and s == 0:
                run.save_model(m, f"agnostic_{v}")
            tf.keras.backend.clear_session(); done += 1
            if done == 1:
                per = time.time() - t0
                run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {total}회 예상 **{per*total/60:.0f}분**")
        for c in CONFIGS:
            run.save_arm(f"{v}_{c}_f{k}", acc[c] / N_SEEDS)
            OOF[f"{v}_{c}"][te_i] = acc[c] / N_SEEDS
        run.log(f"  {v} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")

if sampling_log:
    run.log(f"\nweighted의 실제 샘플링 빈도(겹0): {sampling_log.get('f0', {})}")
    run.log(f"  의도한 분포: {SAMPLE_P}")
run.log(f"\n총 {time.time()-t0:.0f}s")


In [ ]:
# CELL 4 — 【G0 관문】 재사용 arm이 실험10′을 재현하는가
# 정렬이 깨지면 아래 비교가 통째로 거짓이 되므로 여기서 반드시 확인한다.
norm_m = (Y == 0)
GI = {"전두면": [CLASSES.index(c) for c in G_FRONT],
      "혼합":   [CLASSES.index(c) for c in G_MIXED],
      "횡단면": [CLASSES.index(c) for c in G_TRANS]}

for pre in ("fixed", "agno"):
    for c in CONFIGS:
        arr = np.zeros((len(Y), NC))
        for k in range(K_FOLD):
            arr[np.where(CV == k)[0]] = prev_arm(f"{pre}_{c}_f{k}")
        OOF[f"{pre}_{c}"] = arr

def alpha_for(prob, target):
    lo, hi = 0.02, 50.0
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        p = prob.copy(); p[:, 0] *= mid
        if float((p.argmax(1)[norm_m] != 0).mean()) > target: lo = mid
        else: hi = mid
    return hi

ref_fa = float((OOF["fixed_II"].argmax(1)[norm_m] != 0).mean())
MATCH_OK = bool(0.02 < ref_fa < 0.95)
run.log(f"기준 오경보율 = 유도고정 {{II}}의 {ref_fa:.3f}"
        + ("" if MATCH_OK else "  ⛔ 극단 — α=1로 진행"))
PRED = {}
for arm, pr in OOF.items():
    a = alpha_for(pr, ref_fa) if MATCH_OK else 1.0
    p = pr.copy(); p[:, 0] *= a
    PRED[arm] = p.argmax(1)

def mac(arm): return float(f1_score(Y, PRED[arm], average="macro", zero_division=0))
def grp(arm, g):
    f = f1_score(Y, PRED[arm], average=None, labels=range(NC), zero_division=0)
    return float(np.mean([f[i] for i in GI[g]]))

# 실험10′이 보고한 값(하드코딩) — 재현 대조
EXPECT = {"II": -0.0735, "I+II": -0.0121, "II+V1": -0.0164, "12": 0.0013}
run.log("\n【G0】 재사용한 uniform이 실험10′을 재현하는가")
run.log(f"  {'구성':<8}{'재현 손실':>12}{'실험10′':>12}{'차이':>10}")
ok = True
for c in CONFIGS:
    got = mac(f"agno_{c}") - mac(f"fixed_{c}")
    d = got - EXPECT[c]
    if abs(d) > 0.005: ok = False
    run.log(f"  {c:<8}{got:>+12.4f}{EXPECT[c]:>+12.4f}{d:>+10.4f}"
            f"{'' if abs(d) <= 0.005 else '  ❌'}")
if not ok:
    raise RuntimeError("재사용 arm이 실험10′을 재현하지 못했습니다 — 겹 배정이나 "
                       "분할 절차가 어긋났습니다. 비교가 무의미하므로 중단합니다.")
run.log("  ✅ 재현 확인 — 같은 정렬 위에서 비교합니다")


In [ ]:
# CELL 5 — 결과 + 사전등록 채점
ARMS = ["uniform"] + VARIANTS
def arm_key(v, c): return f"agno_{c}" if v == "uniform" else f"{v}_{c}"

run.log("=" * 96)
run.log("【표】 유도고정 대비 손실 (동작점 정합 · 0에 가까울수록 좋다)")
run.log("=" * 96)
run.log(f"  {'구성':<8}{'유도고정':>10}" + "".join(f"{v:>12}" for v in ARMS))
loss = {v: {} for v in ARMS}
for c in CONFIGS:
    fx = mac(f"fixed_{c}")
    row = f"  {c:<8}{fx:>10.4f}"
    for v in ARMS:
        loss[v][c] = mac(arm_key(v, c)) - fx
        row += f"{loss[v][c]:>+12.4f}"
    run.log(row)
run.log("-" * 96)
run.log(f"  {'횡단면군':<8}{'':<10}" + "".join(
    f"{grp(arm_key(v,'II'),'횡단면') - grp('fixed_II','횡단면'):>+12.4f}" for v in ARMS)
    + "   ← {II}에서의 횡단면군 손실")

# ── 붕괴 감시 (실험10-full의 교훈: 특정 클래스가 0으로 죽으면 macro 비교가 거짓말이 된다)
def per_class(arm):
    return f1_score(Y, PRED[arm], average=None, labels=range(NC), zero_division=0)

COLLAPSE = []
run.log("\n【붕괴 감시】 유도고정에서는 살아있는데 변형에서 F1=0 이 된 클래스")
for v in ARMS:
    for c in CONFIGS:
        fxf, vf = per_class(f"fixed_{c}"), per_class(arm_key(v, c))
        for i, cl in enumerate(CLASSES):
            if fxf[i] > 0.05 and vf[i] == 0:
                COLLAPSE.append({"variant": v, "config": c, "class": cl})
                run.log(f"  ❌ {v} · {c} · {cl}  (유도고정 {fxf[i]:.3f} → 0)")
if not COLLAPSE:
    run.log("  ✅ 없음 — macro 비교가 유효하다")

def boot_loss(v_hi, v_lo, cfg, B=BOOT, seed=SEED0):
    """(v_hi 손실) − (v_lo 손실). 양수면 v_hi가 덜 나쁘다.
    유도고정 항은 같은 재표본에서 정확히 상쇄되므로 두 arm만 계산한다."""
    rs = np.random.RandomState(seed); n = len(Y); out = []
    ph, pl = PRED[arm_key(v_hi, cfg)], PRED[arm_key(v_lo, cfg)]
    for _ in range(B):
        i = rs.randint(0, n, n); y = Y[i]
        f_ = lambda p: f1_score(y, p[i], average="macro", zero_division=0)
        out.append(f_(ph) - f_(pl))
    out = np.array(out)
    return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

run.log("\n" + "=" * 96)
run.log("【H1 주가설】 변형 − uniform  ({II}에서 손실이 줄었나 · 양수면 개선)")
run.log("=" * 96)
res = {}
for v in VARIANTS:
    for cfg in ("II", "12"):
        d, l, u = boot_loss(v, "uniform", cfg)
        sig = bool(l > 0 or u < 0)
        res[f"{v}|{cfg}"] = {"delta": d, "ci": [l, u], "significant": sig}
        tag = "★" if sig else " "
        run.log(f"  {v:<10} {cfg:<6} Δ={d:>+8.4f}  [{l:+.4f}, {u:+.4f}] {tag}")

best = max(VARIANTS, key=lambda v: res[f"{v}|II"]["delta"])
H1 = bool(res[f"{best}|II"]["significant"] and res[f"{best}|II"]["delta"] > 0)
H2 = bool(loss[best]["12"] > loss["uniform"]["12"] - 0.02)
H3 = bool(loss[best]["II"] > -0.05)

run.log("\n" + "=" * 96)
run.log("【사전등록 채점】")
run.log("=" * 96)
run.log(f"  H1 {{II}} 손실 개선 유의 → {'✅' if H1 else '❌'} "
        f"(최고 변형 '{best}' Δ={res[f'{best}|II']['delta']:+.4f})")
run.log(f"  H2 {{12}} 희생 없음(−0.02 이내) → {'✅' if H2 else '❌'} "
        f"({best}의 {{12}} 손실 {loss[best]['12']:+.4f} vs uniform {loss['uniform']['12']:+.4f})")
run.log(f"  H3 유도고정 {{II}} 대비 −0.05 이내 → {'✅' if H3 else '❌'} "
        f"({loss[best]['II']:+.4f})")

w_ok = res["weighted|II"]["significant"] and res["weighted|II"]["delta"] > 0
f_ok = res["film|II"]["significant"] and res["film|II"]["delta"] > 0
if w_ok and f_ok:
    mech = "H-데이터와 H-용량 **둘 다** — 샘플링과 특징 공유가 모두 원인이다"
elif w_ok:
    mech = "**H-데이터** — 균등 샘플링이 원인. 어려운 구성을 더 자주 뽑는 것으로 해결된다"
elif f_ok:
    mech = "**H-용량** — 공유 특징이 원인. 마스크가 특징을 변조해야 한다"
else:
    mech = ("어느 쪽도 아니다 — 저유도 손실은 샘플링·용량 문제가 아니라 "
            "**정보량의 본질적 한계**일 수 있다")
run.log(f"  H4 기전 → {mech}")

if H1 and H3:
    verdict = (f"확증 — '{best}'가 {{II}} 손실을 {loss['uniform']['II']:+.4f} → "
               f"{loss[best]['II']:+.4f}로 줄였다. 한 가중치로 웨어러블·텔레메트리를 "
               "함께 서비스하는 전략이 성립한다")
elif H1:
    verdict = (f"부분 확증 — '{best}'가 개선했으나 합격선(−0.05)에는 못 미친다 "
               f"({loss[best]['II']:+.4f}). 두 처방을 결합하거나 distillation을 추가")
else:
    verdict = ("기각 — 샘플링·용량 어느 쪽도 저유도 손실을 못 줄였다. "
               "**배치 전략을 '한 가중치'에서 '구성별 별도 모델'로 바꿔야 한다** "
               "(텔레메트리는 공짜지만 웨어러블은 전용 모델이 필요하다)")

# 붕괴가 있으면 macro 기반 판정 자체가 무효다 — 실험10-full에서 이걸로 한 번 속았다.
best_collapsed = [x for x in COLLAPSE if x["variant"] == best]
if best_collapsed:
    H1 = H2 = H3 = False
    verdict = ("판정 무효 — 최고 변형 '" + best + "'에서 클래스가 붕괴했다("
               + ", ".join(f"{x['config']}·{x['class']}" for x in best_collapsed)
               + "). macro-F1 차이가 '개선'이 아니라 최적화 실패를 재는 중이므로 "
                 "학습 설정(균형비·에폭)을 고친 뒤 다시 돌려야 한다")
run.log(f"\n▶ {verdict}")
run.log("=" * 96)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 3.6))
xs = np.arange(len(CONFIGS))
for i, v in enumerate(ARMS):
    ax.bar(xs + i * 0.26 - 0.26, [loss[v][c] for c in CONFIGS], 0.24, label=v)
ax.axhline(-0.05, ls="--", c="r", lw=1, label="합격선 −0.05")
ax.axhline(0, c="k", lw=0.8)
ax.set_xticks(xs); ax.set_xticklabels(list(CONFIGS))
ax.set_ylabel("유도고정 대비 손실"); ax.set_title("실험9b — RLM 변형별 저유도 손실")
ax.legend(fontsize=8); plt.tight_layout(); run.save_fig("rlm_variants", fig); plt.show()

run.save_json("evaluation", {"loss": loss, "comparisons": res, "sampling_log": sampling_log,
                             "H1": H1, "H2": H2, "H3": H3, "mechanism": mech,
                             "best_variant": best, "verdict": verdict,
                             "collapsed_classes": COLLAPSE})

result = {"week": 2, "exp_id": "exp9b_rlm_fix", "quest": "ailab-2026-0015",
          "task": "lead-agnostic의 저유도 붕괴를 샘플링·FiLM으로 고칠 수 있는가",
          "split": "inter", "metric": "agnostic_minus_fixed_at_II",
          "value": round(loss[best]["II"], 4), "passed": bool(H1 and H3),
          "date": time.strftime("%Y-%m-%d"), "k_fold": K_FOLD, "n_seeds": N_SEEDS,
          "loss": loss, "comparisons": res, "best_variant": best,
          "H1": H1, "H2": H2, "H3": H3, "mechanism": mech, "verdict": verdict,
          "degraded_classes": [f"{x['variant']}·{x['config']}·{x['class']}" for x in COLLAPSE],
          "summary": (f"최고 '{best}' {{II}} 손실 {loss['uniform']['II']:+.4f} → "
                      f"{loss[best]['II']:+.4f} · H1 {H1} H2 {H2} H3 {H3} · "
                      f"{verdict.split(' —')[0]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp9b_rlm_lowlead_fix.ipynb \\
      --quest ailab-2026-0015 --step "exp9b-rlm-lowlead-fix" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")


---

## 결과 읽는 법

| H1 | H3 | 뜻 | 다음 |
|---|---|---|---|
| ✅ | ✅ | **고쳐졌다.** 한 가중치로 웨어러블·텔레메트리를 함께 서비스할 수 있다 | 그 변형을 표준으로 채택. 실험 11(전극위치)·12(도메인)가 이 가중치를 이어받는다 |
| ✅ | ❌ | 개선은 됐으나 합격선 미달 | 두 처방을 **결합**하거나 `{12}` 교사 → `{II}` 학생 distillation 추가 |
| ❌ | — | 샘플링도 용량도 원인이 아니다 | **배치 전략 변경** — 웨어러블은 전용 모델. "한 가중치" 포기의 근거가 된다 |

**❌도 값진 결과입니다.** "한 가중치로 둘 다"가 안 된다는 걸 세 번의 관찰 + 두 번의
처방 실패로 보인 것이라, 구성별 모델을 쓰는 설계 결정에 근거가 생깁니다.

**단, 표를 읽기 전에 【붕괴 감시】부터 봅니다.** 유도고정에서 살아있던 클래스가 변형에서
F1=0 이 되면 macro 차이는 "개선"이 아니라 최적화 실패를 재는 중입니다. 실험10-full이
정확히 그렇게 뒤집혔으므로(HYP 0.441 → 0.088), 붕괴가 잡히면 셀 5가 **판정 무효**를
찍고 H1~H3 를 전부 ❌로 내립니다 — 그때는 해석하지 말고 학습 설정을 고쳐 다시 돌립니다.

## 이 실험이 남기는 것

1. **저유도 손실의 기전** — 데이터(샘플링)인지 용량(특징 공유)인지 갈린다
2. **개선된 agnostic 가중치**(`arms/agnostic_*/weights.keras`) — 실험 11·12가 이어받는다
3. **재사용 패턴의 검증** — 이전 실험의 arm을 읽어 쓰되 G0 관문으로 정렬을 확인하는 절차.
   앞으로 후속 실험의 비용을 절반으로 줄인다

## 한계

- **seed 1개**(실험10′과 맞춤). 변형 간 차이가 작으면 seed 잡음과 구분이 어렵다 —
  H1이 비유의로 나오면 seed를 늘려 재확인할 것.
- `weighted`의 확률 `{II}` 40%는 **임의로 정한 값**이다. 최적화하지 않았다.
  개선이 보이면 그 비율 자체를 튜닝할 여지가 남는다(단, 튜닝은 별도 검증셋에서).
- FiLM은 파라미터를 늘린다(블록마다 `Dense(f)` 2개). 개선이 FiLM 구조 덕인지
  단순히 **용량이 커서**인지는 이 실험으로 갈리지 않는다 → 개선되면 파라미터 수를
  맞춘 대조군이 필요하다.
